# 과제 10 · 11 · 사람 검토 앞에서 멈추고, 받을 수 없는 결정은 다시 묻기

## 현업 시나리오
초안은 정비 기술자가 검토해야 보고서가 됩니다(SOP-EA-01 7.2). 그래프는 검토 앞에서 **실제로 멈춰야** 하고, 결정이 오면 그 자리에서 이어 가야 합니다. 그런데 검사 오류가 있는 초안에 "승인"이 오면 어떻게 할까요? 예외로 거절하면 생각지 못한 일이 생깁니다.

## 학습 목표
- `interrupt` 로 실행을 끊고, checkpointer 에 멈춘 상태를 남긴다.
- `Command(resume=...)` 로 멈춘 지점부터 이어 가고, 앞 단계가 다시 돌지 않는 것을 확인한다.
- 받을 수 없는 결정을 예외로 막으면 건이 굳는 것을 재현하고, 다시 묻는 방식으로 고친다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 검토 앞에서 멈추기

In [ ]:
# prepare 는 초안을 만드는 앞 단계, review 는 사람의 결정을 기다리는 단계입니다.
# interrupt 가 사람에게 보여 줄 packet 을 내보내고 실행을 끊습니다. 멈춘 상태는 checkpointer 에 남습니다.
# 관찰 포인트: invoke 가 끝났는데 report 는 돌지 않았습니다. 다음에 돌 노드가 review 로 남아 있습니다.
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

practice_ran = {"prepare": 0}

class PracticeCase(TypedDict, total=False):
    errors: list
    decision: str
    report: str

def practice_prepare(state):
    practice_ran["prepare"] += 1
    return {"errors": state.get("errors", [])}

def practice_allowed(state):
    return (["escalate", "reject"] if state["errors"] else ["approve", "escalate", "reject"])

def practice_report(state):
    return {"report": "보고서" if state["decision"] in ("approve", "escalate") else ""}

def practice_build(review):
    graph = StateGraph(PracticeCase)
    graph.add_node("prepare", practice_prepare)
    graph.add_node("review", review)
    graph.add_node("report", practice_report)
    graph.add_edge(START, "prepare")
    graph.add_edge("prepare", "review")
    graph.add_edge("review", "report")
    graph.add_edge("report", END)
    return graph.compile(checkpointer=InMemorySaver())

def practice_review_once(state):
    answer = interrupt({"errors": state["errors"], "allowed": practice_allowed(state)})
    return {"decision": answer["decision"]}

practice_graph = practice_build(practice_review_once)
practice_config = {"configurable": {"thread_id": "EVT-2025-0034"}}
practice_paused = practice_graph.invoke({"errors": []}, practice_config)
practice_next = practice_graph.get_state(practice_config).next
print("사람에게 보여 줄 것:", practice_paused["__interrupt__"][0].value)
print("다음에 돌 노드:", practice_next)

**결과 해설** — invoke 가 돌아왔지만 report 는 돌지 않았고, 다음 노드가 review 로 남았습니다. 사람에게 보여 줄 packet(오류 없음, 가능한 결정 셋)은 interrupt 값으로 나옵니다. 멈춘 상태는 thread_id 로 찾을 수 있습니다.

### 2단계 · 결정을 넣어 이어 가기

In [ ]:
# 같은 thread_id 로 Command(resume=결정) 을 넣으면 멈춘 자리부터 이어 갑니다.
# interrupt 를 부른 review 노드는 처음부터 다시 돌고, interrupt 가 이번에는 결정을 돌려줍니다.
# 관찰 포인트: 앞 단계(prepare)는 다시 돌지 않습니다. 실행 횟수가 1 그대로입니다.
practice_done = practice_graph.invoke(Command(resume={"decision": "approve"}), practice_config)
print("결정:", practice_done["decision"], "· 보고서:", practice_done["report"])
practice_prepare_runs = practice_ran["prepare"]
print("prepare 실행 횟수:", practice_prepare_runs)

**결과 해설** — approve 를 넣자 report 까지 가서 보고서가 나왔고, prepare 는 한 번만 돌았습니다. 대신 **review 노드 자체는 재개할 때 처음부터 다시 돕니다.** 그래서 interrupt 앞에는 다시 돌아도 되는 일(packet 만들기)만 둡니다. 알림 전송을 거기 두면 두 번 나갑니다.

### 3단계 · 예외로 막으면 굳는다, 다시 물으면 풀린다

In [ ]:
# 오류가 있는 초안입니다. approve 는 받을 수 없고 escalate 는 받을 수 있습니다.
# 첫 번째 방식: 받을 수 없는 결정이면 예외를 던집니다. 그 뒤에 올바른 결정을 넣어 봅니다.
# 관찰 포인트: 재개 값은 기록에 남아 다시 재생됩니다. 예외를 던지면 올바른 결정도 같은 예외로 끝납니다.
def practice_review_strict(state):
    answer = interrupt({"allowed": practice_allowed(state)})
    if answer["decision"] not in practice_allowed(state):
        raise ValueError(f"{answer['decision']} 은 받을 수 없습니다")
    return {"decision": answer["decision"]}

practice_strict = practice_build(practice_review_strict)
practice_strict_config = {"configurable": {"thread_id": "strict"}}
practice_strict.invoke({"errors": ["근거 목록에 없는 절"]}, practice_strict_config)
practice_stuck = []
for practice_decision in ("approve", "escalate"):
    try:
        practice_strict.invoke(Command(resume={"decision": practice_decision}), practice_strict_config)
        practice_stuck.append((practice_decision, "통과"))
    except ValueError as error:
        practice_stuck.append((practice_decision, str(error)))
print("예외 방식:", practice_stuck)

# 두 번째 방식: 받을 수 없으면 이유를 붙여 다시 묻습니다(interrupt 를 한 번 더).
def practice_review_reask(state):
    answer = interrupt({"allowed": practice_allowed(state)})
    while answer["decision"] not in practice_allowed(state):
        answer = interrupt({"allowed": practice_allowed(state), "message": f"{answer['decision']} 은 받을 수 없습니다"})
    return {"decision": answer["decision"]}

practice_reask = practice_build(practice_review_reask)
practice_reask_config = {"configurable": {"thread_id": "reask"}}
practice_reask.invoke({"errors": ["근거 목록에 없는 절"]}, practice_reask_config)
practice_again = practice_reask.invoke(Command(resume={"decision": "approve"}), practice_reask_config)
practice_reasked = practice_reask.invoke(Command(resume={"decision": "escalate"}), practice_reask_config)
print("다시 묻기:", practice_again["__interrupt__"][0].value["message"], "→", practice_reasked["decision"], practice_reasked["report"])

assert practice_paused["__interrupt__"] and practice_next == ("review",), "검토 앞에서 멈추지 않았습니다"
assert practice_done["report"] == "보고서" and practice_prepare_runs == 1, "앞 단계가 다시 돌았습니다"
assert practice_stuck[1] == ("escalate", "approve 은 받을 수 없습니다"), "올바른 결정을 넣었는데 풀렸습니다(굳는 현상이 재현되지 않음)"
assert practice_reasked["decision"] == "escalate" and practice_reasked["report"] == "보고서"

**결과 해설** — 예외 방식에서는 approve 가 거절된 뒤 escalate 를 넣어도 **"approve 은 받을 수 없습니다"** 로 끝납니다. 처음 넣은 재개 값이 기록에 남아 다시 재생되기 때문입니다. 이 건은 영원히 처리할 수 없게 굳습니다. 다시 묻는 방식에서는 approve 에 이유를 붙인 두 번째 interrupt 가 나오고, escalate 로 보고서까지 갑니다.

## 직접 해 볼 과제
다시 묻는 그래프에 "approve" 를 두 번 넣은 뒤 "reject" 를 넣어 보세요. 몇 번이고 다시 물을 수 있습니까? 실제 앱은 그래프에 닿기 전에 무엇으로 한 번 더 막는지 `app.py` 에서 찾아보세요.

In [ ]:
# 새 thread 로 시작해 결정을 차례로 넣습니다. 받을 수 없는 결정마다 다시 묻습니다.
# 마지막 결과의 report 가 비어 있으면 reject 로 끝난 것입니다.
practice_try_config = {"configurable": {"thread_id": "try"}}
practice_reask.invoke({"errors": ["근거 목록에 없는 절"]}, practice_try_config)
for practice_try_decision in ("approve", "approve", "reject"):
    practice_try = practice_reask.invoke(Command(resume={"decision": practice_try_decision}), practice_try_config)
    print(practice_try_decision, "→", practice_try.get("__interrupt__", [None])[0] and practice_try["__interrupt__"][0].value.get("message"), practice_try.get("report"))

## 중간 결과
멈춘 시점의 packet 과 다음 노드, 재개 뒤 보고서와 앞 단계 실행 횟수, 그리고 예외 방식이 굳는 것과 다시 묻는 방식이 풀리는 것을 확인합니다.

## 실패 경계
checkpointer 가 프로세스 메모리(InMemorySaver)면 재시작 한 번에 검토 대기 중이던 건이 사라집니다. **기다리라고 해 놓고 잊는 것**이라 운영에서는 파일 저장소를 씁니다. 또 저장소에서 상태를 꺼낼 때 허용할 타입을 정하지 않으면 LangGraph 가 경고하고, 다음 버전부터는 꺼내기를 막습니다.

## 실제 app 연결
과제 10 App 의 `review.py::review` 가 같은 다시 묻기 방식으로 멈추고, `review.py::refusal` 이 받을 수 없는 결정의 이유를 만들며, `review.py::allowed_decisions` 가 검사 오류가 있으면 approve 를 뺍니다. `app.py::decide` 는 그래프에 닿기 전에 같은 목록으로 422 를 돌려줍니다. 저장소는 `shared/graph/checkpoint.py::thread_store` 가 고르고, `THREAD_DB` 를 주면 파일에 남아 재시작을 넘깁니다.

## 다음 Notebook 연결
다음 `12_end_to_end_evaluation.ipynb` 에서는 지금까지의 조각을 한 그래프로 잇고, Agent 바깥의 숫자까지 평가합니다.